# XAUUSD M5 Scalping ML Model

**Tujuan:** Prediksi arah scalping M5 XAUUSD — BUY / SELL / WAIT  
**Model:** Stacking ensemble (LGB + XGB + ET → Logistic Regression)  
**Target:** TP=$3 sebelum SL=$2 dalam 3 candle M5 (15 menit ke depan)  
**RR:** 1:1.5

### Perbaikan dari versi lama:
- Target ATR-based (bukan fixed $2/$1)
- M5 sebagai primary, bukan M1
- Walk-forward validation proper
- Fix backtest PnL calculation
- Feature selection ketat (tidak korelasi tinggi)
- Threshold optimasi via Optuna

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
DATA_DIR  = Path('../data')
MODEL_DIR = Path('../models')
MODEL_DIR.mkdir(exist_ok=True)

print('Libraries loaded OK')

Libraries loaded OK


## 1. Load Data

In [2]:
def load_mt5(path):
    df = pd.read_csv(path, sep='\t')
    df.columns = [c.strip().replace('<','').replace('>','').lower() for c in df.columns]
    df['datetime'] = pd.to_datetime(
        df['date'].str.replace('.', '-', regex=False) + ' ' + df['time']
    )
    df = df.sort_values('datetime').reset_index(drop=True).set_index('datetime')
    df = df.drop(columns=['date','time'], errors='ignore')
    df = df.rename(columns={'tickvol':'volume','spread':'spread_pt'})
    for col in ['open','high','low','close']:
        df[col] = df[col].astype(float)
    return df

df = load_mt5(DATA_DIR / 'XAUUSDm_M5.csv')
print(f'M5 candles: {len(df):,}')
print(f'Range: {df.index[0]} → {df.index[-1]}')
print(f'Price range: {df.close.min():.2f} – {df.close.max():.2f}')
df.tail(3)

M5 candles: 100,119
Range: 2024-10-29 17:15:00 → 2026-03-31 17:15:00
Price range: 2539.84 – 5586.78


,open,high,low,close,volume,vol,spread_pt
datetime,,,,,,,
2026-03-31 17:05:00,4647.901,4649.436,4636.916,4639.255,3414,0,280
2026-03-31 17:10:00,4639.171,4645.336,4636.421,4642.992,3534,0,280
2026-03-31 17:15:00,4642.934,4644.997,4641.838,4644.044,706,0,280


## 2. Feature Engineering (M5 Only)

In [3]:
def add_features(df):
    d = df.copy()
    c = d['close']; h = d['high']; l = d['low']; o = d['open']

    # ── Candle structure ────────────────────────────────────────────
    d['body']        = (c - o).abs()
    d['candle_range']= h - l
    d['upper_wick']  = h - d[['close','open']].max(axis=1)
    d['lower_wick']  = d[['close','open']].min(axis=1) - l
    d['body_pct']    = d['body'] / (d['candle_range'] + 1e-9)
    d['is_bull']     = (c > o).astype(int)

    # ── ATR (14) — volatility baseline ──────────────────────────────
    tr = pd.concat([h-l, (h-c.shift()).abs(), (l-c.shift()).abs()], axis=1).max(axis=1)
    d['atr']  = tr.ewm(span=14, adjust=False).mean()
    d['atr5'] = tr.ewm(span=5,  adjust=False).mean()
    d['range_atr'] = d['candle_range'] / (d['atr'] + 1e-9)

    # ── EMAs ────────────────────────────────────────────────────────
    for span in [9, 20, 50, 200]:
        d[f'ema{span}'] = c.ewm(span=span, adjust=False).mean()
    d['price_ema9']  = (c - d['ema9'])  / (d['atr'] + 1e-9)
    d['price_ema20'] = (c - d['ema20']) / (d['atr'] + 1e-9)
    d['price_ema50'] = (c - d['ema50']) / (d['atr'] + 1e-9)
    d['ema9_20']     = (d['ema9'] - d['ema20']) / (d['atr'] + 1e-9)
    d['ema20_50']    = (d['ema20'] - d['ema50']) / (d['atr'] + 1e-9)
    d['trend_ema20'] = (c > d['ema20']).astype(int)
    d['trend_ema50'] = (c > d['ema50']).astype(int)
    d['trend_ema200']= (c > d['ema200']).astype(int)

    # ── RSI ─────────────────────────────────────────────────────────
    delta = c.diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    d['rsi']       = 100 - 100 / (1 + gain / (loss + 1e-9))
    d['rsi_lag1']  = d['rsi'].shift(1)
    d['rsi_slope'] = d['rsi'] - d['rsi_lag1']

    # ── MACD ────────────────────────────────────────────────────────
    ema12 = c.ewm(span=12).mean(); ema26 = c.ewm(span=26).mean()
    d['macd']      = (ema12 - ema26) / (d['atr'] + 1e-9)
    d['macd_sig']  = d['macd'].ewm(span=9).mean()
    d['macd_hist'] = d['macd'] - d['macd_sig']
    d['macd_hist_lag1'] = d['macd_hist'].shift(1)
    d['macd_cross']= np.sign(d['macd_hist']) != np.sign(d['macd_hist_lag1'])
    d['macd_cross']= d['macd_cross'].astype(int)

    # ── ADX ─────────────────────────────────────────────────────────
    dm_p = (h - h.shift()).clip(lower=0)
    dm_n = (l.shift() - l).clip(lower=0)
    di_p = 100 * dm_p.ewm(span=14).mean() / (d['atr'] + 1e-9)
    di_n = 100 * dm_n.ewm(span=14).mean() / (d['atr'] + 1e-9)
    dx   = 100 * (di_p - di_n).abs() / (di_p + di_n + 1e-9)
    d['adx']   = dx.ewm(span=14).mean()
    d['di_diff']= (di_p - di_n) / (di_p + di_n + 1e-9)  # +1=BUY, -1=SELL

    # ── Bollinger Bands ─────────────────────────────────────────────
    bb_m = c.rolling(20).mean(); bb_s = c.rolling(20).std()
    d['bb_pct']  = (c - (bb_m - 2*bb_s)) / (4*bb_s + 1e-9)
    d['bb_width']= (4*bb_s) / (bb_m + 1e-9)

    # ── Stochastic ──────────────────────────────────────────────────
    ll = l.rolling(14).min(); hh = h.rolling(14).max()
    d['stoch_k'] = 100 * (c - ll) / (hh - ll + 1e-9)
    d['stoch_d'] = d['stoch_k'].rolling(3).mean()

    # ── Returns ─────────────────────────────────────────────────────
    for n in [1, 2, 3, 5, 10]:
        d[f'ret{n}'] = c.pct_change(n)

    # ── Momentum ────────────────────────────────────────────────────
    d['mom3']  = c - c.shift(3)
    d['mom5']  = c - c.shift(5)
    d['mom10'] = c - c.shift(10)
    # Normalize by ATR
    d['mom3_atr']  = d['mom3']  / (d['atr'] + 1e-9)
    d['mom5_atr']  = d['mom5']  / (d['atr'] + 1e-9)
    d['mom10_atr'] = d['mom10'] / (d['atr'] + 1e-9)

    # ── Volume ──────────────────────────────────────────────────────
    vol = d['volume'].replace(0, np.nan)
    d['vol_r5']  = vol / (vol.rolling(5).mean()  + 1e-9)
    d['vol_r20'] = vol / (vol.rolling(20).mean() + 1e-9)

    # ── Session flags ───────────────────────────────────────────────
    h_utc = d.index.hour
    d['is_london']  = ((h_utc >= 7)  & (h_utc < 16)).astype(int)
    d['is_ny']      = ((h_utc >= 12) & (h_utc < 21)).astype(int)
    d['is_overlap'] = ((h_utc >= 12) & (h_utc < 16)).astype(int)
    d['hour']       = h_utc
    d['dow']        = d.index.dayofweek

    # ── Candle pattern ──────────────────────────────────────────────
    d['hammer']        = ((d['lower_wick'] > 2*d['body']) & (d['upper_wick'] < d['body'])).astype(int)
    d['shooting_star'] = ((d['upper_wick'] > 2*d['body']) & (d['lower_wick'] < d['body'])).astype(int)
    d['bull_engulf']   = ((d['is_bull']==1) & (d['is_bull'].shift(1)==0) &
                          (c > o.shift(1)) & (o < c.shift(1))).astype(int)
    d['bear_engulf']   = ((d['is_bull']==0) & (d['is_bull'].shift(1)==1) &
                          (o > c.shift(1)) & (c < o.shift(1))).astype(int)

    # ── Swing High/Low (struktur harga) ─────────────────────────────
    d['prev_high'] = h.rolling(10).max().shift(1)
    d['prev_low']  = l.rolling(10).min().shift(1)
    d['near_high'] = (h > d['prev_high'] * 0.999).astype(int)
    d['near_low']  = (l < d['prev_low']  * 1.001).astype(int)

    return d

df = add_features(df)
print(f'Features: {df.shape[1]} kolom')

Features: 68 kolom


## 3. Target Label — ATR-based TP/SL

Label dibuat berdasarkan apakah harga mencapai TP atau SL dalam N candle ke depan.
- **TP = 0.3 × ATR** (scalping target kecil, realistis M5)
- **SL = 0.2 × ATR** (RR 1:1.5)
- **Lookahead = 3 candle M5** (15 menit)
- **Label 2 = BUY wins**, **Label 0 = SELL wins**, **Label 1 = WAIT**

In [4]:
TP_MULT    = 0.3   # TP = 0.3 × ATR
SL_MULT    = 0.2   # SL = 0.2 × ATR  → RR 1:1.5
LOOKAHEAD  = 3     # 3 candle M5 = 15 menit

def make_labels(df, tp_mult=TP_MULT, sl_mult=SL_MULT, lookahead=LOOKAHEAD):
    close  = df['close'].values
    high   = df['high'].values
    low    = df['low'].values
    atr    = df['atr'].values
    n      = len(df)
    labels = np.ones(n, dtype=int)  # default WAIT

    for i in range(n - lookahead):
        entry = close[i]
        tp_d  = atr[i] * tp_mult
        sl_d  = atr[i] * sl_mult
        if tp_d < 0.5 or sl_d < 0.3:   # ATR terlalu kecil, skip
            continue

        buy_tp  = entry + tp_d
        buy_sl  = entry - sl_d
        sell_tp = entry - tp_d
        sell_sl = entry + sl_d

        buy_win = sell_win = False
        for j in range(i+1, i+1+lookahead):
            if not buy_win and not sell_win:
                if high[j] >= buy_tp:
                    buy_win = True; break
                if low[j]  <= buy_sl:
                    break  # BUY SL kena duluan

        for j in range(i+1, i+1+lookahead):
            if not buy_win and not sell_win:
                if low[j]  <= sell_tp:
                    sell_win = True; break
                if high[j] >= sell_sl:
                    break  # SELL SL kena duluan

        if buy_win:
            labels[i] = 2
        elif sell_win:
            labels[i] = 0
        # else: WAIT (1)

    return labels

df['label'] = make_labels(df)

dist = df['label'].value_counts().sort_index()
names = {0:'SELL', 1:'WAIT', 2:'BUY'}
for k,v in dist.items():
    print(f'  {names[k]:4s} ({k}): {v:6,}  ({v/len(df)*100:.1f}%)')

# Drop WAIT untuk training (hanya BUY/SELL)
df_trade = df[df['label'] != 1].copy()
print(f'\nTrading signals: {len(df_trade):,} ({len(df_trade)/len(df)*100:.1f}% dari total)')

  SELL (0): 29,945  (29.9%)
  WAIT (1): 20,057  (20.0%)
  BUY  (2): 50,117  (50.1%)

Trading signals: 80,062 (80.0% dari total)


## 4. Persiapan Data Training

In [5]:
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.model_selection import TimeSeriesSplit

# Kolom yang tidak dipakai sebagai fitur
EXCLUDE = {'open','high','low','close','volume','spread_pt','vol','label',
           'ema9','ema20','ema50','ema200',  # raw price level — data leakage
           'prev_high','prev_low',           # raw price level
           'mom3','mom5','mom10',            # raw price diff — pakai versi ATR-normalized
          }

FEAT_COLS = [c for c in df_trade.columns if c not in EXCLUDE]
print(f'Kandidat fitur: {len(FEAT_COLS)}')

df_clean = df_trade[FEAT_COLS + ['label']].dropna()
print(f'Setelah dropna: {len(df_clean):,} baris')

X = df_clean[FEAT_COLS].values
y = (df_clean['label'] == 2).astype(int).values  # BUY=1, SELL=0 (binary)

# 80/20 time split
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f'\nTrain: {len(X_train):,} | Test: {len(X_test):,}')
print(f'Train BUY: {y_train.mean()*100:.1f}% | Test BUY: {y_test.mean()*100:.1f}%')

Kandidat fitur: 52
Setelah dropna: 80,050 baris

Train: 64,040 | Test: 16,010
Train BUY: 62.7% | Test BUY: 62.1%


In [6]:
# Scaling + Feature Selection
scaler   = RobustScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

selector = SelectKBest(mutual_info_classif, k=40)
X_train_sel = selector.fit_transform(X_train_s, y_train)
X_test_sel  = selector.transform(X_test_s)

selected_feats = [FEAT_COLS[i] for i in selector.get_support(indices=True)]
print('Top 15 fitur terpilih:')
scores = selector.scores_[selector.get_support()]
for f, s in sorted(zip(selected_feats, scores), key=lambda x: -x[1])[:15]:
    print(f'  {f:<25} {s:.4f}')

Top 15 fitur terpilih:
  is_bull                   0.0067
  is_ny                     0.0056
  is_london                 0.0052
  near_high                 0.0050
  trend_ema20               0.0048
  bb_width                  0.0046
  vol_r5                    0.0045
  bb_pct                    0.0038
  hour                      0.0038
  rsi_slope                 0.0037
  mom3_atr                  0.0032
  di_diff                   0.0029
  trend_ema50               0.0028
  near_low                  0.0021
  ema9_20                   0.0021


## 5. Training Model

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (accuracy_score, classification_report,
                             roc_auc_score, precision_score, recall_score, f1_score)

import lightgbm as lgb
import xgboost as xgb
import numpy as np

# ===============================
# 1. HANDLE IMBALANCE
# ===============================
buy_ratio = y_train.mean()
scale_pos = (1 - buy_ratio) / buy_ratio
print(f'Scale pos weight: {scale_pos:.2f}')

# ===============================
# 2. BASE MODELS
# ===============================
lgb_model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    min_child_samples=30,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos,
    random_state=42,
    verbose=-1
)

xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos,
    random_state=42,
    verbosity=0,
    eval_metric='logloss'
)

et_model = ExtraTreesClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=20,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

# ===============================
# 3. STACKING MODEL
# ===============================
stack_model = StackingClassifier(
    estimators=[
        ('lgb', lgb_model),
        ('xgb', xgb_model),
        ('et',  et_model),
    ],
    final_estimator=LogisticRegression(C=0.5, max_iter=1000),
    cv=TimeSeriesSplit(n_splits=5),
    passthrough=False,
    n_jobs=-1
)

print("Training stacking model...")
stack_model.fit(X_train_sel, y_train)

# ===============================
# 4. CALIBRATION (WAJIB 🔥)
# ===============================
print("Calibrating probability...")
calibrated_model = CalibratedClassifierCV(stack_model, method='isotonic', cv=3)
calibrated_model.fit(X_train_sel, y_train)

print("Done!")

# ===============================
# 5. PREDICTION (PAKAI PROBABILITY)
# ===============================
y_proba = calibrated_model.predict_proba(X_test_sel)[:, 1]

# Threshold bisa di-tuning
threshold = 0.6
y_pred = (y_proba > threshold).astype(int)

# ===============================
# 6. EVALUATION (LENGKAP)
# ===============================
print("\n=== MODEL PERFORMANCE ===")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("ROC AUC  :", roc_auc_score(y_test, y_proba))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# ===============================
# 7. BACKTEST (REALISTIC)
# ===============================
balance = 1000
risk = 0.01

df_test = df.iloc[len(df)-len(y_test):].copy()
df_test['proba'] = y_proba
df_test['pred'] = y_pred

for i in range(len(df_test)):
    prob = df_test.iloc[i]['proba']
    
    # hanya ambil signal kuat
    if prob > threshold:
        if df_test.iloc[i]['target'] == 1:
            balance += balance * risk * 1.5
        else:
            balance -= balance * risk

print("\nFinal Balance:", balance)

# ===============================
# 8. SAVE MODEL
# ===============================
import joblib
joblib.dump(calibrated_model, "scalping_model_M5"
""
".pkl")

Scale pos weight: 0.59
Training stacking model...


ValueError: cross_val_predict only works for partitions

## 6. Optimasi Threshold via Optuna

In [ ]:
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    OPTUNA_OK = True
except ImportError:
    OPTUNA_OK = False
    print('optuna tidak ada — pakai threshold default 0.58')

proba_train = stack_model.predict_proba(X_train_sel)[:, 1]
proba_test  = stack_model.predict_proba(X_test_sel)[:, 1]

if OPTUNA_OK:
    def objective(trial):
        thr = trial.suggest_float('threshold', 0.50, 0.75)
        pred = (proba_train >= thr).astype(int)
        # Optimasi precision × recall (tidak asal precision tinggi tapi coverage 0)
        prec = precision_score(y_train, pred, zero_division=0)
        rec  = recall_score(y_train, pred, zero_division=0)
        if rec < 0.05: return 0.0  # harus ada coverage minimal
        return prec * rec  # F1 proxy

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=100, show_progress_bar=False)
    THRESHOLD = study.best_params['threshold']
    print(f'Threshold optimal: {THRESHOLD:.3f} (score: {study.best_value:.4f})')
else:
    THRESHOLD = 0.58

y_pred = (proba_test >= THRESHOLD).astype(int)

print(f'\n=== TEST SET RESULT (threshold={THRESHOLD:.3f}) ===')
print(f'Accuracy : {accuracy_score(y_test, y_pred)*100:.2f}%')
print(f'Precision: {precision_score(y_test, y_pred, zero_division=0)*100:.2f}%')
print(f'Recall   : {recall_score(y_test, y_pred, zero_division=0)*100:.2f}%')
print(f'F1 Score : {f1_score(y_test, y_pred, zero_division=0)*100:.2f}%')
print(f'AUC      : {roc_auc_score(y_test, proba_test):.4f}')

# Coverage = seberapa banyak sinyal di atas threshold
coverage = y_pred.mean() * 100
print(f'Coverage : {coverage:.1f}% sinyal diambil')

## 7. Walk-Forward Validation

In [ ]:
from sklearn.base import clone

tscv     = TimeSeriesSplit(n_splits=5)
wf_aucs  = []
wf_precs = []

for fold, (tr_idx, val_idx) in enumerate(tscv.split(X)):
    X_tr, X_val = X[tr_idx], X[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    sc_  = RobustScaler()
    X_tr_s  = sc_.fit_transform(X_tr)
    X_val_s = sc_.transform(X_val)

    sel_ = SelectKBest(mutual_info_classif, k=40)
    X_tr_sel  = sel_.fit_transform(X_tr_s, y_tr)
    X_val_sel = sel_.transform(X_val_s)

    # Hanya LGB untuk speed (proxy walk-forward)
    bw = (1-y_tr.mean())/y_tr.mean()
    m  = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05,
                             scale_pos_weight=bw, random_state=42, verbose=-1)
    m.fit(X_tr_sel, y_tr)
    prob = m.predict_proba(X_val_sel)[:, 1]
    pred = (prob >= THRESHOLD).astype(int)

    auc  = roc_auc_score(y_val, prob) if len(np.unique(y_val)) > 1 else 0.5
    prec = precision_score(y_val, pred, zero_division=0)
    wf_aucs.append(auc)
    wf_precs.append(prec)
    print(f'  Fold {fold+1}: AUC={auc:.4f}  Precision={prec*100:.1f}%  '
          f'Signals={pred.sum()}/{len(pred)}')

print(f'\n  Mean AUC      : {np.mean(wf_aucs):.4f} ± {np.std(wf_aucs):.4f}')
print(f'  Mean Precision: {np.mean(wf_precs)*100:.1f}% ± {np.std(wf_precs)*100:.1f}%')

## 8. Backtest Realistis

In [ ]:
# Backtest hanya di test set (bukan train!)
test_df   = df_clean.iloc[split:].copy()
test_df['prob_buy'] = proba_test
test_df['signal']   = (proba_test >= THRESHOLD).astype(int)

# TP/SL dalam USD berdasarkan ATR aktual (bukan estimasi flat)
INITIAL_CAPITAL = 100.0
LOT_SIZE        = 0.01   # 0.01 lot XAUUSD = $0.1/pips ≈ $1/USD

capital = INITIAL_CAPITAL
trades  = []

for i, (idx, row) in enumerate(test_df[test_df['signal']==1].iterrows()):
    atr     = float(row['atr'])
    entry   = float(row['close'])
    tp_dist = atr * TP_MULT
    sl_dist = atr * SL_MULT
    actual  = int(row['label'])  # 2=BUY wins, 0=SELL wins, 1=WAIT

    if actual == 2:   # BUY signal correct
        pnl_pts = tp_dist
        outcome = 'WIN'
    elif actual == 0: # BUY signal wrong (SELL wins)
        pnl_pts = -sl_dist
        outcome = 'LOSS'
    else:             # WAIT — assume SL kena (konservatif)
        pnl_pts = -sl_dist
        outcome = 'LOSS'

    # PnL dalam USD: 1 USD move = $1 per 0.01 lot XAUUSD
    pnl_usd = pnl_pts * LOT_SIZE * 100   # 0.01 lot × 100 = $1/USD
    capital += pnl_usd

    trades.append({
        'time'    : idx,
        'entry'   : entry,
        'tp_dist' : round(tp_dist, 2),
        'sl_dist' : round(sl_dist, 2),
        'pnl_usd' : round(pnl_usd, 4),
        'capital' : round(capital, 4),
        'outcome' : outcome,
    })

trades_df = pd.DataFrame(trades)

if len(trades_df) > 0:
    wins  = (trades_df['outcome']=='WIN').sum()
    total = len(trades_df)
    wr    = wins/total*100
    gross_win  = trades_df[trades_df['pnl_usd']>0]['pnl_usd'].sum()
    gross_loss = trades_df[trades_df['pnl_usd']<0]['pnl_usd'].abs().sum()
    pf    = gross_win / (gross_loss + 1e-9)
    net   = trades_df['pnl_usd'].sum()

    # Max drawdown
    cap_series = trades_df['capital']
    rolling_max = cap_series.cummax()
    dd = (cap_series - rolling_max) / rolling_max * 100
    max_dd = dd.min()

    print(f'=== BACKTEST (0.01 lot, ${INITIAL_CAPITAL} modal) ===')
    print(f'  Total trade   : {total}')
    print(f'  Win rate      : {wr:.1f}%  ({wins}W / {total-wins}L)')
    print(f'  Profit factor : {pf:.2f}')
    print(f'  Net PnL       : ${net:.2f}')
    print(f'  Final capital : ${capital:.2f}')
    print(f'  Max drawdown  : {max_dd:.1f}%')
    print(f'  Avg TP dist   : ${trades_df["tp_dist"].mean():.2f}')
    print(f'  Avg SL dist   : ${trades_df["sl_dist"].mean():.2f}')
else:
    print('Tidak ada trade di test set — turunkan THRESHOLD')

In [ ]:
# Equity Curve
if len(trades_df) > 0:
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))

    # Equity
    axes[0].plot(trades_df['time'], trades_df['capital'], color='steelblue', linewidth=1.5)
    axes[0].axhline(INITIAL_CAPITAL, color='gray', linestyle='--', alpha=0.5)
    axes[0].set_title(f'Equity Curve — {total} trades | WR {wr:.1f}% | PF {pf:.2f} | Net ${net:.2f}')
    axes[0].set_ylabel('Capital ($)')
    axes[0].grid(alpha=0.3)

    # PnL distribution
    colors = ['green' if x > 0 else 'red' for x in trades_df['pnl_usd']]
    axes[1].bar(range(len(trades_df)), trades_df['pnl_usd'], color=colors, alpha=0.7, width=0.8)
    axes[1].axhline(0, color='black', linewidth=0.8)
    axes[1].set_title('PnL per Trade')
    axes[1].set_ylabel('PnL ($)')
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('../results/equity_curve_v2.png', dpi=150, bbox_inches='tight')
    plt.show()

## 9. Simpan Model

In [ ]:
import joblib, json

bundle = {
    'model'         : stack_model,
    'scaler'        : scaler,
    'selector'      : selector,
    'feature_cols'  : FEAT_COLS,
    'selected_feats': selected_feats,
    'prob_threshold': THRESHOLD,
    'tp_mult'       : TP_MULT,
    'sl_mult'       : SL_MULT,
    'lookahead'     : LOOKAHEAD,
}
joblib.dump(bundle, MODEL_DIR / 'xauusd_scalping_model_M5.joblib', compress=3)

meta = {
    'symbol'        : 'XAUUSDm',
    'timeframe'     : 'M5',
    'model_type'    : 'Stacking(LGB+XGB+ET->LR)',
    'accuracy'      : round(accuracy_score(y_test, y_pred)*100, 2),
    'precision'     : round(precision_score(y_test, y_pred, zero_division=0)*100, 2),
    'recall'        : round(recall_score(y_test, y_pred, zero_division=0)*100, 2),
    'f1'            : round(f1_score(y_test, y_pred, zero_division=0)*100, 2),
    'auc'           : round(roc_auc_score(y_test, proba_test), 4),
    'prob_threshold': round(THRESHOLD, 4),
    'tp_mult'       : TP_MULT,
    'sl_mult'       : SL_MULT,
    'rr_ratio'      : round(TP_MULT/SL_MULT, 1),
    'lookahead_bars': LOOKAHEAD,
    'n_features'    : len(selected_feats),
    'top_features'  : selected_feats[:10],
    'n_train'       : len(X_train),
    'n_test'        : len(X_test),
    'backtest': {
        'trades'    : total,
        'win_rate'  : round(wr, 2),
        'profit_factor': round(pf, 3),
        'net_pnl_usd'  : round(net, 4),
        'max_dd_pct'   : round(max_dd, 2),
    } if len(trades_df) > 0 else {},
}
with open(MODEL_DIR / 'xauusd_scalping_model_M5_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('Model disimpan ke:', MODEL_DIR / 'xauusd_scalping_model_M5.joblib')
print('Meta  disimpan ke:', MODEL_DIR / 'xauusd_scalping_model_M5_meta.json')
print(json.dumps(meta, indent=2))

## 10. Update predictor.py agar cocok dengan fitur M5-only

Karena sekarang model hanya pakai M5 (bukan M1+M5), update `predictor.py` untuk predict dari M5 saja.

In [ ]:
# Cek fitur yang diperlukan vs yang ada di live bot
print('Fitur yang dibutuhkan model:')
for i, f in enumerate(selected_feats, 1):
    print(f'  {i:2d}. {f}')